# Data Preparation for Machine Learning

This notebook prepares our cleaned crime dataset for the five Machine Learning models we will build: Linear Regression, Decision Tree, Random Forest, KNN, and K-Means.

**Important Note:** This notebook strictly handles data splitting, feature engineering, and scaling. We will not train any ML models here.

## Step 1 — Load and Clean Data

We load the `01_District_wise_crimes_committed_IPC_2001_2012.csv` dataset. As discovered in Phase 1, we must remove the summary rows (`TOTAL` and `DELHI UT TOTAL`) to prevent massive double-counting. We will also perform quick verifications.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
file_path = '../data/factors/crime/01_District_wise_crimes_committed_IPC_2001_2012.csv'
df = pd.read_csv(file_path)

# Remove summary rows
df_clean = df[~df['DISTRICT'].isin(['TOTAL', 'DELHI UT TOTAL'])].copy()

# Verify data quality
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")

# Check for negative values in numeric columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
negative_counts = (df_clean[numeric_cols] < 0).sum().sum()
print(f"Negative values: {negative_counts}")

# Check for infinite values
infinite_counts = np.isinf(df_clean[numeric_cols]).sum().sum()
print(f"Infinite values: {infinite_counts}")

Shape after cleaning: (8597, 33)
Missing values: 0
Duplicate rows: 0
Negative values: 0
Infinite values: 0


## Step 2 — Create Project-Defined Crime Scores

To analyze crime meaningfully without target leakage, we will create two **project-defined composite scores**. 
*Note: These are custom analytical metrics for this project, not official government crime indices.*

1. **Violent Crime Score:** `MURDER` + `ATTEMPT TO MURDER` + `RAPE` + `KIDNAPPING & ABDUCTION`
2. **Property Crime Score:** `THEFT` + `BURGLARY` + `ROBBERY`

*Note: `TOTAL IPC CRIMES` remains in the dataframe for reference but will strictly **not** be used as an ML predictor.*

In [2]:
# Create Violent Crime Score
df_clean['Violent Crime Score'] = (
    df_clean['MURDER'] + 
    df_clean['ATTEMPT TO MURDER'] + 
    df_clean['RAPE'] + 
    df_clean['KIDNAPPING & ABDUCTION']
)

# Create Property Crime Score
df_clean['Property Crime Score'] = (
    df_clean['THEFT'] + 
    df_clean['BURGLARY'] + 
    df_clean['ROBBERY']
)

print("Composite scores created successfully!")
display(df_clean[['DISTRICT', 'YEAR', 'Violent Crime Score', 'Property Crime Score']].head())

Composite scores created successfully!


,DISTRICT,YEAR,Violent Crime Score,Property Crime Score
0,ADILABAD,2001,257,438
1,ANANTAPUR,2001,352,573
2,CHITTOOR,2001,244,974
3,CUDDAPAH,2001,178,275
4,EAST GODAVARI,2001,221,1483


## Step 3 — Time-Based Train/Test Split

Our dataset contains repeated observations for districts over multiple years. 
If we used a standard random split, we might put a district's 2010 data into the training set, and its 2005 data into the testing set. This would allow the model to "peek" into the future.

To prevent this temporal leakage, we use a strict chronological split:
* **Training Data:** 2001–2009
* **Testing Data:** 2010–2012

*(Note: While this solves temporal leakage, it does not perfectly eliminate all geographic idiosyncrasies, but it is the most robust beginner-friendly approach).*

In [3]:
# Split the data chronologically
train_df = df_clean[df_clean['YEAR'] <= 2009].copy()
test_df = df_clean[df_clean['YEAR'] >= 2010].copy()

print(f"Training Data: Years {train_df['YEAR'].min()} to {train_df['YEAR'].max()} | Shape: {train_df.shape}")
print(f"Testing Data: Years {test_df['YEAR'].min()} to {test_df['YEAR'].max()} | Shape: {test_df.shape}")

Training Data: Years 2001 to 2009 | Shape: (6321, 35)
Testing Data: Years 2010 to 2012 | Shape: (2276, 35)


## Step 4 — Create Classification Target (`Violent Crime Level`)

Our classification models will predict the `Violent Crime Level` (Low / Medium / High).
We will use statistical tertiles (33rd and 67th percentiles) of the `Violent Crime Score`.

**CRITICAL RULE:** We calculate these thresholds using **ONLY the training data**. We then apply those exact thresholds to classify both the training and testing sets. If we used the test data to calculate the thresholds, we would be leaking future information into our target definition.

In [4]:
# Calculate thresholds from TRAINING data only
lower_threshold = train_df['Violent Crime Score'].quantile(0.33)
upper_threshold = train_df['Violent Crime Score'].quantile(0.67)

print(f"Lower Threshold (33rd percentile): {lower_threshold}")
print(f"Upper Threshold (67th percentile): {upper_threshold}\n")

# Function to apply labels
def categorize_crime(score):
    if score <= lower_threshold:
        return 'Low'
    elif score <= upper_threshold:
        return 'Medium'
    else:
        return 'High'

# Apply to both sets
train_df['Violent Crime Level'] = train_df['Violent Crime Score'].apply(categorize_crime)
test_df['Violent Crime Level'] = test_df['Violent Crime Score'].apply(categorize_crime)

# Verify Class Distributions
print("--- TRAINING SET CLASS DISTRIBUTION ---")
train_counts = train_df['Violent Crime Level'].value_counts()
train_pct = train_df['Violent Crime Level'].value_counts(normalize=True) * 100
display(pd.DataFrame({'Count': train_counts, 'Percentage (%)': train_pct.round(2)}))

print("\n--- TESTING SET CLASS DISTRIBUTION ---")
test_counts = test_df['Violent Crime Level'].value_counts()
test_pct = test_df['Violent Crime Level'].value_counts(normalize=True) * 100
display(pd.DataFrame({'Count': test_counts, 'Percentage (%)': test_pct.round(2)}))

print("\nObservation: The testing set maintains a healthy representation of all three classes using the historical thresholds.")

Lower Threshold (33rd percentile): 79.0
Upper Threshold (67th percentile): 168.0

--- TRAINING SET CLASS DISTRIBUTION ---


,Count,Percentage (%)
Violent Crime Level,,
Medium,2137,33.81
Low,2102,33.25
High,2082,32.94



--- TESTING SET CLASS DISTRIBUTION ---


,Count,Percentage (%)
Violent Crime Level,,
High,900,39.54
Medium,716,31.46
Low,660,29.00



Observation: The testing set maintains a healthy representation of all three classes using the historical thresholds.


## Step 5 — Prepare Linear Regression Data

**Target:** `Violent Crime Score`
**Predictors:** `Property Crime Score`, `CHEATING`, `RIOTS`

*(We strictly exclude `MURDER`, `RAPE`, etc., because they mathematically construct the target. We exclude `TOTAL IPC CRIMES` to prevent general target leakage).*

In [5]:
reg_features = ['Property Crime Score', 'CHEATING', 'RIOTS']
reg_target = 'Violent Crime Score'

X_train_reg = train_df[reg_features]
y_train_reg = train_df[reg_target]

X_test_reg = test_df[reg_features]
y_test_reg = test_df[reg_target]

print("Linear Regression datasets prepared!")

Linear Regression datasets prepared!


## Step 6 — Prepare Classification Data

This data will be used by the Decision Tree, Random Forest, and KNN classifiers.

**Target:** `Violent Crime Level`
**Features:** `THEFT`, `BURGLARY`, `ROBBERY`, `CHEATING`, `RIOTS`

*(We use raw property crime columns here instead of the composite score to let the Tree models find specific patterns. We strictly exclude violent crimes and totals to prevent target leakage).*

In [6]:
clf_features = ['THEFT', 'BURGLARY', 'ROBBERY', 'CHEATING', 'RIOTS']
clf_target = 'Violent Crime Level'

X_train_clf = train_df[clf_features]
y_train_clf = train_df[clf_target]

X_test_clf = test_df[clf_features]
y_test_clf = test_df[clf_target]

print("Classification datasets prepared!")

Classification datasets prepared!


## Step 7 — Prepare K-Means Clustering Data

K-Means will group districts according to their **historical crime profiles**. 
Instead of treating "Pune 2005" and "Pune 2006" as separate entities, we will calculate the average historical crime count for each district across all available years.

**Features to average and cluster:** 
`MURDER`, `ATTEMPT TO MURDER`, `RAPE`, `KIDNAPPING & ABDUCTION`, `THEFT`, `BURGLARY`, `ROBBERY`, `RIOTS`, `CHEATING`

*(We strictly exclude `DISTRICT`, `STATE/UT`, `YEAR`, `TOTAL IPC CRIMES`, and our custom labels).*

Because these crimes have vastly different numerical magnitudes (e.g., Theft is in the thousands, Murder in the tens), we **must** apply `StandardScaler` so that K-Means calculates distances fairly.

In [7]:
kmeans_features = [
    'MURDER', 'ATTEMPT TO MURDER', 'RAPE', 'KIDNAPPING & ABDUCTION', 
    'THEFT', 'BURGLARY', 'ROBBERY', 'RIOTS', 'CHEATING'
]

# 1. Group by District and calculate historical average
kmeans_df = df_clean.groupby('DISTRICT')[kmeans_features].mean().reset_index()

# 2. Extract only the features for scaling
X_kmeans_raw = kmeans_df[kmeans_features]

# 3. Apply StandardScaler
scaler = StandardScaler()
X_kmeans_scaled = scaler.fit_transform(X_kmeans_raw)

# Convert back to DataFrame for readability later
X_kmeans = pd.DataFrame(X_kmeans_scaled, columns=kmeans_features)

print("K-Means historical profiles generated and scaled!")

K-Means historical profiles generated and scaled!


## Step 8 — Final Verification & Data Preparation Summary

Let's verify the shapes of all our prepared datasets before we save or pass them to the ML notebooks.

In [8]:
print("=== LINEAR REGRESSION ===")
print(f"X_train_reg: {X_train_reg.shape} | y_train_reg: {y_train_reg.shape}")
print(f"X_test_reg:  {X_test_reg.shape} | y_test_reg:  {y_test_reg.shape}\n")

print("=== CLASSIFICATION (Tree, Forest, KNN) ===")
print(f"X_train_clf: {X_train_clf.shape} | y_train_clf: {y_train_clf.shape}")
print(f"X_test_clf:  {X_test_clf.shape} | y_test_clf:  {y_test_clf.shape}\n")

print("=== K-MEANS CLUSTERING ===")
print(f"Data shape: {X_kmeans.shape} (One row per district)")
print(f"Total Districts: {kmeans_df.shape[0]}")
print(f"Features: {list(X_kmeans.columns)}\n")

print("--- Sample of Scaled K-Means Data (First 3 rows) ---")
display(X_kmeans.head(3))

=== LINEAR REGRESSION ===
X_train_reg: (6321, 3) | y_train_reg: (6321,)
X_test_reg:  (2276, 3) | y_test_reg:  (2276,)

=== CLASSIFICATION (Tree, Forest, KNN) ===
X_train_clf: (6321, 5) | y_train_clf: (6321,)
X_test_clf:  (2276, 5) | y_test_clf:  (2276,)

=== K-MEANS CLUSTERING ===
Data shape: (806, 9) (One row per district)
Total Districts: 806
Features: ['MURDER', 'ATTEMPT TO MURDER', 'RAPE', 'KIDNAPPING & ABDUCTION', 'THEFT', 'BURGLARY', 'ROBBERY', 'RIOTS', 'CHEATING']

--- Sample of Scaled K-Means Data (First 3 rows) ---


,MURDER,ATTEMPT TO MURDER,RAPE,KIDNAPPING & ABDUCTION,THEFT,BURGLARY,ROBBERY,RIOTS,CHEATING
0,3.713448,1.579942,4.709622,4.596875,1.718626,-0.473986,0.746063,2.676931,0.449593
1,3.266941,-0.567522,5.732185,2.794884,1.358514,-0.406628,0.096008,4.752993,0.102803
2,-0.749548,-0.512874,-0.878173,-0.581803,-0.369416,-0.197780,-0.436026,-0.538560,-0.392674


## Data Preparation Summary

* **Linear Regression:** Prepared continuous targets (`Violent Crime Score`) and non-overlapping continuous predictors (`Property Crime Score`, `CHEATING`, `RIOTS`), split chronologically.
* **Decision Tree & Random Forest:** Prepared categorical targets (`Violent Crime Level`) derived from training tertiles, with raw property/disturbance crime counts as predictors, split chronologically.
* **KNN:** Will use the exact same classification datasets. (Note: Scaling will be applied directly inside the KNN notebook to prevent leakage between train/test).
* **K-Means:** Grouped the raw data by district to create historical averages of 9 major crimes, and applied standard scaling to ensure fair distance calculations across vastly different crime magnitudes.